# Claim review v2: draft decisions

Read docs/SEMANTIC_CLAIM_GUIDELINE_v2_DRAFT.md first. Run 05 -> 06 -> 07 for prerequisites. These proposals remain pending owner confirmation. Source material, v1 worksheets and frozen annotations are preserved.


In [ ]:
from pathlib import Path
import sys, hashlib, json
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/processed/model_corpus_v1.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or notebooks directory.")
sys.path.insert(0, str(ROOT / "src"))
from claim_adjudication import read_table, validate_proposals, load_or_create, export_review
base = ROOT / "data/processed"
source = base / "model_corpus_v1.csv"
expected = "d66a330b121ae5e59d96b58fb12d34b0c4fdbd9059d427abdc7bd182b07bcc2c"
if hashlib.sha256(source.read_bytes()).hexdigest() != expected:
    raise ValueError("Frozen corpus differs.")
screen_manifest = json.loads((base / "semantic_screen_manifest_v1.json").read_text())
sample_manifest = json.loads((base / "semantic_development_sample_manifest_v1.json").read_text())
for path, expected_hash in [(ROOT / "src/semantic_screening.py", screen_manifest["screening_module_sha256"]),
                            (base / "semantic_candidate_screen_v1.csv", sample_manifest["screening_table_sha256"]),
                            (base / "annotation_development_v2.csv", sample_manifest["annotation_source_sha256"])]:
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f"Prerequisite changed: {path.name}")
if screen_manifest["source_sha256"] != expected or sample_manifest["source_sha256"] != expected:
    raise ValueError("Source manifest differs.")
corpus = read_table(source)
selection = read_table(base / "semantic_development_sample_v1.csv")
published = json.loads((ROOT / "reports/semantic_development_sampling_v1.json").read_text())
ids = selection.execution_number.tolist()
if ids != published["execution_ids_in_display_order"] or ids != sample_manifest["execution_ids_in_display_order"]:
    raise ValueError("Fixed selection differs.")
for row in selection.itertuples():
    original = corpus.set_index("execution_number").loc[row.execution_number]
    if row.model_text != original.model_text or row.source_url != original.source_url:
        raise ValueError("Selected context differs.")
proposals = read_table(ROOT / "reports/semantic_development_proposals_v2.csv")
if not set(proposals.execution_number).issubset(ids):
    raise ValueError("Proposal outside sample.")
validate_proposals(proposals, corpus)
worksheet_path = base / "semantic_development_adjudication_v2.csv"
worksheet = load_or_create(proposals, worksheet_path)
documents_path = base / "semantic_document_decisions_v2.csv"
if not documents_path.exists():
    documents = pd.DataFrame({"execution_number":ids})
    for field in ["owner_document_decision", "owner_note", "reviewer_id", "reviewed_at"]:
        documents[field] = ""
    with documents_path.open("x", encoding="utf-8", newline="") as handle:
        documents.to_csv(handle,index=False)
additions = base / "semantic_owner_additions_v2.csv"
if not additions.exists():
    with additions.open("x", encoding="utf-8", newline="") as handle:
        pd.DataFrame(columns=["addition_id", "execution_number", "evidence_span", "selected_predicate", "review_note"]).to_csv(handle,index=False)
print(f"Verified {len(proposals)} draft propositions, source evidence and 21 fixed IDs.")
print("V2 worksheets ready; v1 and existing owner edits were not overwritten.")
display(worksheet[["claim_id", "execution_number", "owner_decision"]])


## Review without changing evidence

Open the two v2 decision CSVs in a UTF-8 capable editor. Allowed claim decisions: accept / revise / defer; blank means pending. All decisions need reviewer_id and reviewed_at (ISO with timezone). Revise requires all five owner fields and a note; defer requires a note. Document decisions: reviewed_selected / reviewed_no_selection / defer, each with note and reviewer/time. They cover selected material only, never certify exhaustive claim annotation. Use the separate additions CSV for missing propositions; send additions before exporting. Rerun to validate, never edit evidence cells.


In [ ]:
def show_v2(execution_number):
    print(corpus.set_index("execution_number").loc[execution_number, "model_text"])
    display(proposals.loc[proposals.execution_number.eq(execution_number)])
show_v2(231)


In [ ]:
# Call only after completing and checking both owner worksheets.
# If the additions CSV has rows, send it for versioning first.
def export_completed_review(tag):
    if not read_table(additions).empty:
        raise ValueError("Owner additions need a new proposals version before export.")
    return export_review(ROOT, proposals, worksheet_path, documents_path, tag)
# export_completed_review("v2_owner_review_01")
